<h4> Zadanie1: Pobierz część danych Amazon Fine Foods review (z github). Zbuduj model regresji logistycznej wieloklasowej na podstawie reprezentacji BoW, SoW lub TF-IDF.
    
    
- podziel dane na zbiór treningowy i testowy (8:2)
- naucz model na zbiorze treningowym
- wyznacz dokładność na zbiorze treningowym i testowym po etapie uczenia; powinieneś otrzymać min. 60% na zbiorze zbiorze testowym.
    
Potestuj różne topolgie sieci/funkcje aktywacji/wymiary.

1. Pierwszy sposób - TF-IDF dawał najlepsze wyniki w porównaniu z SoW i BoW

In [ ]:
import pandas as pd
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#prepare data
with open('food_reviews.txt', 'r') as file:
    foods = file.read().lower()

df = pd.DataFrame([line.split(";", 1) for line in foods.splitlines()], columns=["score", "review"])

df = df.dropna()  #delete NA
df["score"] = df["score"].astype(int) #score as int

#train/test set
X_train, X_test, y_train, y_test = train_test_split(df["review"], df["score"], test_size=0.2, random_state=42)

#TF-IDF (words to number vectors)
tfidf = TfidfVectorizer(max_features=25000, ngram_range=(1,5))  #added penta-grams (best results) #max features, check how many istotnych słów,
X_train_tfidf = tfidf.fit_transform(X_train)            #teraz ngramy są obiektem na ktr ustalamy max dieatures
X_test_tfidf = tfidf.transform(X_test)

#model
model = LogisticRegression(max_iter=1500, multi_class='multinomial', solver='lbfgs')  # softmax
model.fit(X_train_tfidf, y_train)

#acc
train_acc = accuracy_score(y_train, model.predict(X_train_tfidf))
test_acc = accuracy_score(y_test, model.predict(X_test_tfidf))

print(f"Dokładność treningowa: {train_acc:.2%}")
print(f"Dokładność testowa: {test_acc:.2%}")

c:\Users\kajaw\miniconda3\envs\Pbioinf1.2\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Dokładność treningowa: 82.03%
Dokładność testowa: 58.64%


2. Drugi sposób - przekształcenia liniowe

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

#TF-IDF to tensors
X_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32) #text features
y_tensor = torch.tensor(y_train.values, dtype=torch.long) #class tags, long - całkowite

n_features = X_tensor.shape[1] #number of features
n_classes = len(set(y_train)) #number of classes

# Definicja modelu
class Model(nn.Module):
    def __init__(self, n_input_features, h1):
        super().__init__()
        self.linear1 = nn.Linear(n_input_features, h1) #hidden layer
        self.linear3 = nn.Linear(h1, n_classes)

    def forward(self, x):
        x = F.elu(self.linear1(x))
        x = self.linear3(x)
        return x  # CrossEntropyLoss zawiera softmax

# Parametry
h1 = 7
model = Model(n_features, h1)

# Uczenie
num_epochs = 3000
learning_rate = 1
loss_function = nn.CrossEntropyLoss() #for multiclass: LogSoftmax + NLLLoss

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) #stochastic gradient descent
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5) #decrease learning rate if no imporvement

#training
for epoch in range(num_epochs):
    model.train()  #training mode
    optimizer.zero_grad()  #gradient cleaning

    y_pred = model(X_tensor) #output - logits
    loss = loss_function(y_pred, y_tensor) #loss function - average error between predicted and expected

    loss.backward() #find gradients
    optimizer.step() #update weights

    scheduler.step(loss)  #scheduler - update learning rate based on loss

    if (epoch + 1) % 100 == 0:  #print outcomes
        print(f'epoch: {epoch + 1}, loss = {loss.item():.4f}')

#test
with torch.no_grad(): #without calculating gradients
    y_predicted = model(X_tensor)
    y_predicted_cls = y_predicted.argmax(1) #predict class
    acc = y_predicted_cls.eq(y_tensor).sum() / float(y_tensor.shape[0])
    print(f'Train accuracy: {acc.item():.4f}')

X_test_tensor = torch.tensor(X_test_tfidf.toarray(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    y_test_pred_cls = y_test_pred.argmax(1)
    test_acc = y_test_pred_cls.eq(y_test_tensor).sum() / float(y_test_tensor.shape[0])
    print(f'Test accuracy: {test_acc.item():.4f}')

epoch: 100, loss = 1.5870
epoch: 200, loss = 1.4601
epoch: 300, loss = 1.3146
epoch: 400, loss = 1.2172
epoch: 500, loss = 1.1716
epoch: 600, loss = 1.1312
epoch: 700, loss = 1.0950
epoch: 800, loss = 1.0620
epoch: 900, loss = 1.0316
epoch: 1000, loss = 1.0032
epoch: 1100, loss = 0.9764
epoch: 1200, loss = 0.9508
epoch: 1300, loss = 0.9275
epoch: 1400, loss = 0.9155
epoch: 1500, loss = 0.9037
epoch: 1600, loss = 0.8921
epoch: 1700, loss = 0.8806
epoch: 1800, loss = 0.8693
epoch: 1900, loss = 0.8581
epoch: 2000, loss = 0.8471
epoch: 2100, loss = 0.8361
epoch: 2200, loss = 0.8253
epoch: 2300, loss = 0.8146
epoch: 2400, loss = 0.8041
epoch: 2500, loss = 0.7936
epoch: 2600, loss = 0.7832
epoch: 2700, loss = 0.7729
epoch: 2800, loss = 0.7627
epoch: 2900, loss = 0.7525
epoch: 3000, loss = 0.7425
Train accuracy: 0.7341
Test accuracy: 0.5784


Zadanie2: Wybierz dowolny zestaw tekstów piosenek (co najmniej 6) i na ich podstawie zbuduj model generujący kilka kolejnych słów piosenki ( kolejne słowo, za każdym razem w oparciu o 2 (lub 3) poprzednie słowa). 


Potestuj różne rozmiary embeddingów, zmodyfikuj topologię sieci jeżeli to konieczne.

In [179]:
#download lyrics
import requests
from bs4 import BeautifulSoup

def lyrics_download(author, title):
    url = f"https://www.tekstowo.pl/piosenka,{author},{title}.html"
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        lyrics = soup.find("div", class_="song-text").text.strip()
        return lyrics
    except:
        print(f"Failed to find {title} by {author}.")

#list of fisrt 100 songs by The doors available on tekstowo
songs = ['_you_need_meet__don_t_get_no_further', 'people_are_strange', 'a_feast_of_friends', 'a_little_game', 'across_the_sea', 'adolf_hitler', 'alabama_song__whisky_bar_', 'american_night', 'an_american_prayer', 'angels_and_sailors', 'awake', 'awake_ghost_song', 'away_in_india', 'baby_please_don_t_go', 'babylon_fading', 'back_door_man', 'been_down_so_long', 'bird_of_prey', 'black_polished_chrome', 'blue_sunday', 'break_on_through__to_the_other_side_', 'build_me_a_woman', 'carol', 'cars_hiss_by_my_window', 'close_to_you', 'crawling_king_snake', 'crossroads', 'curses__invocation', 'dawn_s_highway', 'dead_cats__dead_rats', 'do_it', 'don_t_go_no_farther', 'down_on_the_farm', 'easy_ride', 'end_of_the_night', 'ensenada', 'fever', 'five_to_one', 'four_billion_souls', 'get_out_of_my_life_woman', 'get_up_and_dance', 'ghost_song', 'gloria', 'goin__to_new_york', 'good_rockin_', 'graveyard_poem', 'hang_on_to_your_life', 'hardwood_floor', 'heartbreak_hotel', 'hello__i_love_you', 'horse_latitudes_', 'house_announcer', 'hyacinth_house', 'i_can_t_see_your_face_in_my_mind', 'i_looked_at_you', 'i_will_never_be_untrue', 'i_m_a_king_bee', 'i_m_a_man', 'i_m_horny__i_m_stoned', 'i_m_your_doctor', 'in_the_eye_of_the_sun', 'indian_summer', 'it_slipped_my_mind', 'l_a__woman', 'l_america', 'lament', 'land_ho', 'light_my_fire', 'lions_in_the_street', 'little_red_rooster', 'love_her_madly', 'love_hides', 'love_me_tender', 'love_me_two_times', 'love_street', 'mack_the_knife', 'maggie_m_gill', 'mean_mustard_blues', 'mental_floss', 'money', 'moonlight_drive', 'my_eyes_have_seen_you', 'my_wild_love', 'mystery_train', 'names_of_the_kingdom', 'newborn_awakening', 'no_me_moleste_mosquitoes', 'not_to_touch_the_earth', 'orange_county_suite', 'paris_blues', 'peace_frog', 'people_get_ready', 'petition_the_lord_with_prayer', 'queen_of_the_highway', 'queen_of_the_magazines', 'riders_on_the_storm', 'roadhouse_blues', 'rock_is_dead', 'rock_me_baby']

lyr = ""
for s in songs:
    lyr += lyrics_download('the_doors', s)

filtered_lines = [line for line in lyr.splitlines() if not (
    line.startswith("Tekst") or line.startswith("Dodaj") or line.startswith("Historia") 
)]

with open("lyrics.txt", "w") as f:
    for line in filtered_lines:
        if line.strip():  #omit empty lines
            f.write(line + "\n")

In [222]:
#prepare data
import nltk
from nltk import word_tokenize
from nltk import ngrams
import string

with open('lyrics.txt', 'r') as file:
    lyrics = file.read()

lyrics = word_tokenize(lyrics.lower())
lyrics = [word for word in lyrics if word not in string.punctuation]
print(len(lyrics), lyrics[:10])
lyrics = lyrics[:5000]

16271 ['you', 'need', 'meat', 'go', 'to', 'the', 'market', 'you', 'need', 'bread']


In [223]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

#unique words
vocab = list(set(lyrics))

#assign id to unique words
word_to_ix = {word: i for i, word in enumerate(vocab)}

#create 4-grams
N4_GM = list(ngrams(lyrics, 4))
N4_GM = [([x,y,z], q) for x,y,z,q in N4_GM] #change formatting

train_data, test_data = train_test_split(N4_GM, test_size=0.2, random_state=42)

#model
class NGramModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)  #words to embeddings 
        self.linear1 = nn.Linear(context_size*embedding_dim, HD) #1st linear transformation (with hidden dimention)
        self.linear2 = nn.Linear(HD, vocab_size)  #2nd linear transforamtion

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1)) #embeddings
        out = F.elu(self.linear1(embeds))  #elu on the output of 1st transf.
        out = self.linear2(out)             #transform to linear
        log_probs = F.log_softmax(out, dim=1) #probab logarithms (soft_max)
        return log_probs
    
#parameters
CS = 3 #number of context words
ED = 32 #embedding vector size
HD = 64 #hidden dimention size

learning_rate = 0.01
loss_function = nn.NLLLoss()  
model = NGramModel(len(vocab), ED, CS) 
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5) #decrease learning rate if no imporvement

#model evaluation
print(model.eval()) 

#training
for epoch in range(100):
    for context, target in train_data:
        model.train()
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)  #data
        log_probs = model(context_idxs)                                                 
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long)) #loss function
        
        optimizer.zero_grad()   #gradient zeroing     
        loss.backward()         #find gradient
        optimizer.step()        #update parameters 
    scheduler.step(loss)        #scheduler - update learning rate based on loss
       
    if (epoch+1) % 10 == 0:
        print(f'epoch: {epoch+1}, loss = {loss.item():.4f}') 

NGramModel(
  (embeddings): Embedding(1081, 32)
  (linear1): Linear(in_features=96, out_features=64, bias=True)
  (linear2): Linear(in_features=64, out_features=1081, bias=True)
)
epoch: 10, loss = 5.0063
epoch: 20, loss = 1.8566
epoch: 30, loss = 1.5736
epoch: 40, loss = 1.3861
epoch: 50, loss = 1.2954
epoch: 60, loss = 1.2278
epoch: 70, loss = 1.2042
epoch: 80, loss = 1.1979
epoch: 90, loss = 1.1922
epoch: 100, loss = 1.1839


In [ ]:
#check accuracy
def evaluate(model, data, word_to_ix, label="Accuracy"):
    model.eval()
    correct = 0
    with torch.no_grad():
        for context, target in data:
            context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)
            output = model(context_idxs)
            if torch.argmax(output).item() == word_to_ix[target]:
                correct += 1
    acc = correct / len(data)
    print(f"{label}: {acc:.2%}")

#Test
evaluate(model, train_data, word_to_ix, "Training Accuracy")
evaluate(model, test_data, word_to_ix, "Test Accuracy")
#to increase test accuracy, use more the whole songs list 

Training Accuracy: 94.07%
Test Accuracy: 39.30%


In [225]:
#generate words
def generate_text(model, word_to_ix, start_words, length=15):
    import numpy as np
    for i in range(length):
        with torch.no_grad():
            context  = start_words[-3:]
            while len(context) < 3:
                context.insert(0, list(word_to_ix.keys())[0])
            context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long) 
            pred = model(context_idxs) 
            index_of_prediction = np.argmax(pred)  #find index with the highest value
            start_words.append(vocab[index_of_prediction])   #find word with found index
    return start_words

print(generate_text(model, word_to_ix, start_words=['live', 'in', 'the']))
print(generate_text(model, word_to_ix, start_words=['live']))


['live', 'in', 'the', 'light', 'of', 'certain', 'south', 'cruel', 'bindings', 'the', 'servants', 'have', 'the', 'power', 'dog-men', 'their', 'mean', 'women']
['live', 'we', 'have', 'death', 'i', 'straddle', 'the', 'fence', 'and', 'my', 'balls', 'hurt', 'well', 'show', 'me', 'the']


Zadanie3: Zbuduj model, który będzie przewidywał słowo w oparciu o kontekst - 2 wcześniejsze i 2 następne słowa. Zastosuj go do cytatów (plik quotes.txt).
    
Potestuj różne rozmiary embeddingów, zmodyfikuj topologię sieci jeżeli to konieczne.


In [226]:
import string
import nltk
from nltk import word_tokenize
from nltk import ngrams

with open('quotes.txt', 'r') as file:
    quotes = file.read().lower()
    
quotes = word_tokenize(quotes)
quotes = [q for q in quotes if q not in string.punctuation]

#dict
vocab = list(set(quotes))
word_to_ix = {word: i for i, word in enumerate(vocab)} #word to id dict
ix_to_word = {i: word for word, i in word_to_ix.items()} #id to word dict

#5grams
N5_GM = list(ngrams(quotes, 5))
N5_GM = [([x, y, u, v], z) for x, y, z, u, v in N5_GM]

train_data, test_data = train_test_split(N5_GM, test_size=0.2, random_state=42)

In [ ]:
#model
class NGramModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)  #words to embeddings 
        self.linear1 = nn.Linear(context_size*embedding_dim, HD) #1st linear transformation (with hidden dimention)
        self.linear2 = nn.Linear(HD, vocab_size)  #2nd linear transforamtion

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view(1, -1) #embeddings
        out = F.elu(self.linear1(embeds))  #elu on the output of 1st transf.
        out = self.linear2(out)             #transform to linear
        log_probs = F.log_softmax(out, dim=1) #probab logarithms (soft_max)
        return log_probs
    
#parameters
CS = 4 #number of context words
ED = 20 #embedding vector size
HD = 40 #hidden dimention size

learning_rate = 0.001
loss_function = nn.NLLLoss()  
model = NGramModel(len(vocab), ED, CS) 
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5) #decrease learning rate if no imporvement

#model evaluation
print(model.eval()) 

#training
for epoch in range(160):
    for context, target in train_data:
        model.train()
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)  #data
        log_probs = model(context_idxs)                                                 
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long)) #loss function
        
        optimizer.zero_grad()   #gradient zeroing     
        loss.backward()         #find gradient
        optimizer.step()        #update parameters 
    scheduler.step(loss)        #scheduler - update learning rate based on loss
       
    if (epoch+1) % 20 == 0:
        print(f'epoch: {epoch+1}, loss = {loss.item():.4f}') 

NGramModel(
  (embeddings): Embedding(6348, 20)
  (linear1): Linear(in_features=80, out_features=40, bias=True)
  (linear2): Linear(in_features=40, out_features=6348, bias=True)
)
epoch: 20, loss = 2.9499
epoch: 40, loss = 2.6966
epoch: 60, loss = 2.5384
epoch: 80, loss = 2.4620
epoch: 100, loss = 2.4284
epoch: 120, loss = 2.4062
epoch: 140, loss = 2.3927
epoch: 160, loss = 2.3841


In [234]:
#accuracy
evaluate(model, train_data, word_to_ix, "Training Accuracy")
evaluate(model, test_data, word_to_ix, "Test Accuracy")

Training Accuracy: 24.16%
Test Accuracy: 15.15%


In [237]:
#word prediction
def predict_middle(context_words):
    with torch.no_grad():
        idxs = torch.tensor([word_to_ix[w] for w in context_words], dtype=torch.long)
        output = model(idxs)
        predicted_idx = torch.argmax(output).item()
        return ix_to_word[predicted_idx]


example = ['women', 'seem', 'when', 'you']  #wicked
print(predict_middle(example))

and
